## Section 1: Setup & Imports


In [1]:
import os
import sys
from pathlib import Path
import json
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from PIL import Image
from tqdm.notebook import tqdm
import pickle
from datetime import datetime

# Resolve repository root from notebook location
# Notebook path: <repo>/notebooks/certify/isic_certify.ipynb -> ROOT = parent.parent
ROOT = Path.cwd().resolve().parent.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Import certification pipeline
from src.models.factory import get_model
from src.xai.attribution_unified import (
    IntegratedGradientsUnified,
    GradCAMUnified,
    RISEUnified,
    OcclusionUnified,
    LRPUnified
)
from src.certify.smoothing import RandomizedSmoothingAttributor
from src.certify.evaluate import CertificationEvaluator

# Setup
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'PyTorch version: {torch.__version__}')
print(f'Repo root: {ROOT}')

# Paths (use repo-root absolute paths)
CHECKPOINT_DIR = ROOT / 'notebooks' / 'output' / 'checkpoints' / 'isic'
HEATMAP_DIR = ROOT / 'notebooks' / 'output' / 'attributions_viz' / 'isic'  # optional
CERTIFY_DIR = ROOT / 'notebooks' / 'output' / 'certifications' / 'isic'
CERTIFY_DIR.mkdir(parents=True, exist_ok=True)

print(f"Checkpoint dir: {CHECKPOINT_DIR}")
print(f"Heatmap dir: {HEATMAP_DIR}")
print(f"Certification output dir: {CERTIFY_DIR}")

Device: cpu
PyTorch version: 2.9.1+cpu
Repo root: D:\git projects\certified-attribution-medical-imaging
Checkpoint dir: D:\git projects\certified-attribution-medical-imaging\notebooks\output\checkpoints\isic
Heatmap dir: D:\git projects\certified-attribution-medical-imaging\notebooks\output\attributions_viz\isic
Certification output dir: D:\git projects\certified-attribution-medical-imaging\notebooks\output\certifications\isic


In [2]:
# Paper-recommended certification parameters
CERT_CONFIG = {
    'sigma': 0.15,          # Gaussian noise std
    'tau': 0.75,            # Certification threshold
    'num_samples': 100,     # Smoothing samples (n)
    'batch_size': 16,       # Processing batch size
    'alpha': 0.001,         # Significance level (Clopper-Pearson)
    'k_percents': [50, 25, 5]  # Sparsification levels K
}

# Attribution methods to evaluate
ATTR_METHODS = ['IntegratedGradients', 'GradCAM', 'RISE', 'Occlusion', 'LRP']

# Grid evaluation setup
GRID_CONFIG = {
    'grid_size': 2,         # 2x2 grid
    'num_grids': 5,         # 5 grids per model (for visualization)
    'target_cell': (0, 0)   # Top-left cell (ground truth)
}

print('=== Certification Parameters (Paper Eq. 5-7) ===')
for k, v in CERT_CONFIG.items():
    print(f'  {k}: {v}')
print(f'\nAttribution methods: {ATTR_METHODS}')
print(f'Grid evaluation: {GRID_CONFIG}')

# Compute certified radius R = σ * Φ^(-1)(τ)
from scipy.special import ndtri
certified_radius = CERT_CONFIG['sigma'] * ndtri(CERT_CONFIG['tau'])
print(f'\n=> Certified radius R = σ·Φ⁻¹(τ) = {certified_radius:.4f}')
print(f'   Guarantees robustness for ||δ||₂ < {certified_radius:.4f}')

=== Certification Parameters (Paper Eq. 5-7) ===
  sigma: 0.15
  tau: 0.75
  num_samples: 100
  batch_size: 16
  alpha: 0.001
  k_percents: [50, 25, 5]

Attribution methods: ['IntegratedGradients', 'GradCAM', 'RISE', 'Occlusion', 'LRP']
Grid evaluation: {'grid_size': 2, 'num_grids': 5, 'target_cell': (0, 0)}

=> Certified radius R = σ·Φ⁻¹(τ) = 0.1012
   Guarantees robustness for ||δ||₂ < 0.1012


## Critical Paper Implementation Fixes

**Two major correctness issues have been fixed:**

### 1. **Gradient Computation for Attribution Methods**

- **Issue**: `torch.no_grad()` was wrapping attribution computations, breaking gradient-based methods (IntegratedGradients, GradCAM, LRP)
- **Fix**: Removed `torch.no_grad()` and wrapped each attribution call with `torch.enable_grad()` in `src/certify/smoothing.py`
- **Impact**: This was causing degenerate heatmaps and incorrect certification results (e.g., "100% certified everywhere")

### 2. **High-Confidence Filtering**

- **Issue**: Certifying all images regardless of model prediction confidence or correctness
- **Paper requirement**: Only certify images where:
  - Model prediction matches ground truth label: `pred == label`
  - Prediction confidence is high: `softmax(pred) >= 0.8`
- **Fix**: Added pre-certification filtering in both notebook and `certify_isic_server.py`
- **Impact**: Now matches paper methodology - only certifies reliable, confident predictions

These fixes ensure the implementation correctly follows the paper's approach and produces valid certification results.


In [3]:
# Discover available models in checkpoint directory
checkpoint_dir = CHECKPOINT_DIR

available_models = []
if checkpoint_dir.exists():
    # Find model subdirectories and checkpoint files (.pt/.pth)
    model_dirs = [d for d in checkpoint_dir.iterdir() if d.is_dir()]
    for d in sorted(model_dirs):
        model_name = d.name
        # Search for checkpoint files recursively
        candidates = list(d.rglob('*.pt')) + list(d.rglob('*.pth'))
        ckpt_path = None
        if candidates:
            # Pick the most recently modified
            candidates.sort(key=lambda p: p.stat().st_mtime, reverse=True)
            ckpt_path = str(candidates[0])
        available_models.append({
            'name': model_name,
            'checkpoint': ckpt_path
        })
    print(f'✓ Found {len(available_models)} models:')
    for m in available_models:
        print(f"  - {m['name']} (ckpt: {'found' if m['checkpoint'] else 'missing'})")
else:
    print(f'⚠ Checkpoint directory not found: {checkpoint_dir}')
    print('Please ensure trained model checkpoints exist under output/checkpoints/isic/')

print(f'\nTotal models to certify: {len(available_models)}')

✓ Found 6 models:
  - densenet121 (ckpt: found)
  - efficientnet_b0 (ckpt: found)
  - efficientnet_b1 (ckpt: found)
  - mobilenet_v2 (ckpt: found)
  - resnet18 (ckpt: found)
  - resnet50 (ckpt: found)

Total models to certify: 6


In [ ]:
# Storage for results (supports resume)
import pickle

# Try to load existing results (full or partial)
certification_results = {}
loaded_from = None

# First, try to load full saved results (with timestamp)
output_dir = CERTIFY_DIR
pkl_files = sorted(output_dir.glob('results_*.pkl'))
# Exclude partial results from this list
full_pkl_files = [f for f in pkl_files if 'partial' not in f.name]

if full_pkl_files:
    latest_full = full_pkl_files[-1]
    try:
        with open(latest_full, 'rb') as f:
            certification_results = pickle.load(f)
        loaded_from = latest_full
        print(f'✓ Loaded full saved results: {latest_full.name}')
    except Exception as e:
        print(f'⚠ Failed to load full results: {e}')

# If no full results, try partial
if not certification_results:
    partial_path = CERTIFY_DIR / 'results_partial.pkl'
    if partial_path.exists():
        try:
            with open(partial_path, 'rb') as f:
                certification_results = pickle.load(f)
            loaded_from = partial_path
            print(f'✓ Loaded partial results: {partial_path.name}')
        except Exception as e:
            print(f'⚠ Failed to load partial results: {e}')

if not certification_results:
    print('✓ Starting fresh results store')
else:
    # Display what was loaded
    num_models = len(certification_results)
    num_total_entries = sum(
        len(entries) 
        for model_data in certification_results.values() 
        for method_data in model_data.values() 
        for entries in method_data.values()
    )
    print(f'  → {num_models} model(s) with {num_total_entries} total certification entries')

# Load ISIC validation dataset
from torchvision import transforms
from torch.utils.data import DataLoader
from src.datasets.isic import ISICDataset

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

val_dir = ROOT / 'data' / 'raw' / 'isic'
val_dataset = ISICDataset(val_dir, split='val', transform=val_transform)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)

print(f'✓ Loaded validation dataset: {len(val_dataset)} images')

print('\n' + '='*80)
print('CERTIFICATION PIPELINE (Eq. 4-7)')
print('='*80)


def is_done(model_name, method_name, k_percent, image_idx):
    if model_name not in certification_results:
        return False
    if method_name not in certification_results[model_name]:
        return False
    if k_percent not in certification_results[model_name][method_name]:
        return False
    return any(entry.get('image_idx') == image_idx for entry in certification_results[model_name][method_name][k_percent])


def save_partial():
    partial_path = CERTIFY_DIR / 'results_partial.pkl'
    partial_path.parent.mkdir(parents=True, exist_ok=True)
    with open(partial_path, 'wb') as f:
        pickle.dump(certification_results, f)


for model_info in tqdm(available_models, desc='Models'):
    model_name = model_info['name']
    print(f"\n{'='*80}")
    print(f"Model: {model_name}")
    print(f"{'='*80}")
    
    # Load model
    try:
        use_pretrained = model_info['checkpoint'] is None
        model, cfg = get_model(model_name, num_classes=8, pretrained=use_pretrained, device=DEVICE)
        if model_info['checkpoint']:
            checkpoint = torch.load(model_info['checkpoint'], map_location=DEVICE)
            state = checkpoint.get('model_state_dict', checkpoint)
            model.load_state_dict(state)
        model.eval()
        print(f"✓ Model ready ({'pretrained' if use_pretrained else 'checkpoint loaded'})")
    except Exception as e:
        print(f'✗ Failed to load model: {e}')
        continue
    
    if model_name not in certification_results:
        certification_results[model_name] = {}
    
    # Initialize attribution methods
    print(f'Initializing attribution methods...')
    
    def get_target_layer(model, model_name):
        """Get last conv layer for Grad-CAM."""
        if 'resnet' in model_name.lower():
            return model.layer4[-1].conv2 if hasattr(model.layer4[-1], 'conv2') else model.layer4[-1]
        elif 'densenet' in model_name.lower():
            return model.features.denseblock4
        elif 'efficientnet' in model_name.lower():
            return list(model.features.modules())[-1]
        elif 'mobilenet' in model_name.lower():
            return list(model.features.modules())[-1]
        else:
            for name, module in reversed(list(model.named_modules())):
                if isinstance(module, nn.Conv2d):
                    return module
            raise ValueError(f'Could not find target layer for {model_name}')
    
    try:
        target_layer = get_target_layer(model, model_name)
        
        attribution_methods = {
            'IntegratedGradients': IntegratedGradientsUnified(model, DEVICE),
            'GradCAM': GradCAMUnified(model, target_layer, DEVICE),
            'RISE': RISEUnified(model, DEVICE),
            'Occlusion': OcclusionUnified(model, DEVICE),
            'LRP': LRPUnified(model, DEVICE, epsilon=1e-6)
        }
        print(f'✓ All 5 attribution methods initialized')
    except Exception as e:
        print(f'✗ Failed to initialize methods: {e}')
        continue
    
    # Create smoother
    smoother = RandomizedSmoothingAttributor(model, None, device=DEVICE)
    
    # Certification per method (using first 5 validation images)
    num_test_images = 5
    high_confidence_threshold = 0.8  # Paper: only certify high-confidence predictions
    
    for img_idx, batch in enumerate(val_loader):
        if img_idx >= num_test_images:
            break
            
        # Dataset returns a dict: {'image': tensor, 'label': int/tensor, 'meta': {...}}
        test_image = batch['image'].to(DEVICE)
        label_value = batch['label']
        label_int = int(label_value.squeeze().item()) if isinstance(label_value, torch.Tensor) else int(label_value)
        
        # PAPER REQUIREMENT: Only certify high-confidence correct predictions
        with torch.no_grad():
            logits = model(test_image)
            probs = torch.softmax(logits, dim=1)
            pred_class = logits.argmax(dim=1).item()
            pred_confidence = probs[0, pred_class].item()
        
        # Skip if prediction is wrong or low confidence
        if pred_class != label_int:
            print(f'\\n  Image {img_idx+1}/{num_test_images} SKIPPED (pred={pred_class} != label={label_int})')
            continue
        if pred_confidence < high_confidence_threshold:
            print(f'\\n  Image {img_idx+1}/{num_test_images} SKIPPED (confidence={pred_confidence:.3f} < {high_confidence_threshold})')
            continue
        
        # Ensure gradients on input for attribution methods
        test_image.requires_grad_(True)
        
        print(f'\\n  Image {img_idx+1}/{num_test_images} (label: {label_int}, pred: {pred_class}, conf: {pred_confidence:.3f})')
        
        for method_name in tqdm(ATTR_METHODS, desc=f'  Methods', leave=False):
            if method_name not in attribution_methods:
                print(f'    Skipping {method_name} (not in methods)')
                continue
            
            attr_func = attribution_methods[method_name]
            
            if method_name not in certification_results[model_name]:
                certification_results[model_name][method_name] = {}
            
            # Certify for each K value
            for k_percent in CERT_CONFIG['k_percents']:
                if is_done(model_name, method_name, k_percent, img_idx):
                    # Already done (resume scenario)
                    continue
                try:
                    target_h, target_w = test_image.shape[-2:]

                    # Create a fresh attribution function wrapper with grad enabled and spatially aligned output
                    def attr_wrapper(img, target_class=label_int):
                        img.requires_grad_(True)
                        with torch.enable_grad():
                            heat = attr_func.attribute(img, target_class=target_class)
                        
                        # Convert to numpy and ensure correct shape
                        if isinstance(heat, torch.Tensor):
                            heat_np = heat.detach().cpu().numpy()
                        else:
                            heat_np = np.array(heat)
                        
                        # Remove batch dimension if present
                        while heat_np.ndim > 2:
                            if heat_np.shape[0] == 1:
                                heat_np = heat_np.squeeze(0)
                            elif heat_np.shape[-1] == 1:
                                heat_np = heat_np.squeeze(-1)
                            else:
                                # Multi-channel, average over channels
                                if heat_np.ndim == 3:
                                    heat_np = heat_np.mean(axis=0)
                                elif heat_np.ndim == 4:
                                    heat_np = heat_np.mean(axis=(0, 1))
                                break
                        
                        # Ensure 2D
                        if heat_np.ndim != 2:
                            raise ValueError(f'Cannot reduce attribution to 2D, shape: {heat_np.shape}')
                        
                        # Resize to target spatial size if needed
                        if heat_np.shape != (target_h, target_w):
                            from scipy.ndimage import zoom
                            zoom_factors = (target_h / heat_np.shape[0], target_w / heat_np.shape[1])
                            heat_np = zoom(heat_np, zoom_factors, order=1)
                        
                        return heat_np

                    smoother.attribution_func = attr_wrapper

                    # Run certification (Eq. 5-7) with visualization artifacts stored
                    results = smoother.certify(
                        test_image,
                        k_percent=k_percent,
                        target_class=label_int,
                        sigma=CERT_CONFIG['sigma'],
                        num_samples=CERT_CONFIG['num_samples'],
                        tau=CERT_CONFIG['tau'],
                        batch_size=CERT_CONFIG['batch_size'],
                        alpha=CERT_CONFIG['alpha'],
                        save_noisy_samples=True,
                        max_noisy_samples=3
                    )

                    # Store results with image index
                    if k_percent not in certification_results[model_name][method_name]:
                        certification_results[model_name][method_name][k_percent] = []
                    certification_results[model_name][method_name][k_percent].append({
                        'image_idx': img_idx,
                        'label': label_int,
                        'results': results
                    })

                    # Save incrementally to allow resume on errors
                    save_partial()

                except Exception as e:
                    import traceback
                    print(f'    ✗ K={k_percent}% failed: {e}')
                    print(f'       Traceback: {traceback.format_exc()[-200:]}')
            
            print(f'    ✓ {method_name}')

print(f"\n✓ Certification complete for {len(certification_results)} models")
# Final save
save_partial()
print(f"✓ Partial results saved to {CERTIFY_DIR / 'results_partial.pkl'}")

✓ Loaded partial results: results_partial.pkl
  → 1 model(s) with 6 total certification entries
✓ Loaded validation dataset: 522 images

CERTIFICATION PIPELINE (Eq. 4-7)


Models:   0%|          | 0/6 [00:00<?, ?it/s]


Model: densenet121
✓ Model ready (checkpoint loaded)
Initializing attribution methods...
✓ All 5 attribution methods initialized

  Image 1/5 (label: 0)


  Methods:   0%|          | 0/5 [00:00<?, ?it/s]

    ✓ IntegratedGradients
    ✓ GradCAM


KeyboardInterrupt: 

In [8]:
# Save results to pickle and JSON
import json
import pickle
from datetime import datetime


def compute_metrics_summary(cert_results):
    """Aggregate per-image certification metrics and means."""
    summary = {}
    for model_name, methods_dict in cert_results.items():
        summary[model_name] = {}
        for method_name, k_dict in methods_dict.items():
            summary[model_name][method_name] = {}
            for k_percent, entries in k_dict.items():
                if not entries:
                    continue
                per_image = []
                for entry in entries:
                    res = entry.get('results', {})
                    if 'certified_map' not in res:
                        continue
                    c_map = res['certified_map']
                    total = c_map.size
                    per_image.append({
                        'image_idx': entry.get('image_idx'),
                        'label': entry.get('label'),
                        'pct_certified': float(np.sum(c_map != -1) / total * 100.0),
                        'pct_abstained': float(np.sum(c_map == -1) / total * 100.0),
                        'pct_certified_1': float(np.sum(c_map == 1) / total * 100.0),
                        'pct_certified_0': float(np.sum(c_map == 0) / total * 100.0),
                        'certified_radius': float(res.get('certified_radius', 0.0)),
                    })
                if not per_image:
                    continue
                summary[model_name][method_name][k_percent] = {
                    'per_image': per_image,
                    'mean': {
                        'pct_certified': float(np.mean([p['pct_certified'] for p in per_image])),
                        'pct_abstained': float(np.mean([p['pct_abstained'] for p in per_image])),
                        'pct_certified_1': float(np.mean([p['pct_certified_1'] for p in per_image])),
                        'pct_certified_0': float(np.mean([p['pct_certified_0'] for p in per_image])),
                        'certified_radius': float(np.mean([p['certified_radius'] for p in per_image])),
                    }
                }
    return summary


metrics_summary = compute_metrics_summary(certification_results)

# Use the globally defined CERTIFY_DIR
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
pkl_path = CERTIFY_DIR / f'results_{timestamp}.pkl'
json_path = CERTIFY_DIR / f'results_{timestamp}.json'

# Pickle the full results (numpy arrays preserved)
with open(pkl_path, 'wb') as f:
    pickle.dump(certification_results, f)
print(f'✓ Saved pickle: {pkl_path}')

# JSON summary (means only; arrays omitted for compactness)
json_summary = {}
for model_name, methods_dict in certification_results.items():
    json_summary[model_name] = {}
    for method_name, k_dict in methods_dict.items():
        json_summary[model_name][method_name] = {}
        for k_percent, entries in k_dict.items():
            if not entries:
                continue
            if k_percent not in metrics_summary.get(model_name, {}).get(method_name, {}):
                continue
            mean_metrics = metrics_summary[model_name][method_name][k_percent]['mean']
            json_summary[model_name][method_name][str(k_percent)] = {
                'num_images': len(entries),
                'pct_certified': mean_metrics['pct_certified'],
                'pct_abstained': mean_metrics['pct_abstained'],
                'pct_certified_1': mean_metrics['pct_certified_1'],
                'pct_certified_0': mean_metrics['pct_certified_0'],
                'certified_radius': mean_metrics['certified_radius'],
                'alpha': float(CERT_CONFIG['alpha'])
            }

with open(json_path, 'w') as f:
    json.dump(json_summary, f, indent=2)
print(f'✓ Saved JSON: {json_path}')

print(f'\n✓ Results saved for {len(json_summary)} models')

✓ Saved pickle: D:\git projects\certified-attribution-medical-imaging\notebooks\output\certifications\isic\results_20251229_000400.pkl
✓ Saved JSON: D:\git projects\certified-attribution-medical-imaging\notebooks\output\certifications\isic\results_20251229_000400.json

✓ Results saved for 1 models


In [9]:
# Load previous results (optional)
previous_results = None

# List available pickle files
pkl_files = sorted(CERTIFY_DIR.glob('results_*.pkl'))

if pkl_files:
    latest_pkl = pkl_files[-1]
    print(f'Found previous results: {latest_pkl.name}')
    with open(latest_pkl, 'rb') as f:
        previous_results = pickle.load(f)
    print(f'✓ Loaded previous results from {latest_pkl}')
else:
    print('No previous results found. Using current session results.')

Found previous results: results_partial.pkl
✓ Loaded previous results from D:\git projects\certified-attribution-medical-imaging\notebooks\output\certifications\isic\results_partial.pkl


In [12]:
# Compute evaluation metrics (Eq. 6) using per-image results
if 'metrics_summary' not in globals():
    metrics_summary = compute_metrics_summary(certification_results)

print('\n' + '='*80)
print('METRICS TABLE (per-image aggregated)')
print('='*80)

for model_name in sorted(metrics_summary.keys())[:1]:  # Show first model as sample
    print(f'\n{model_name}:')
    for method_name in ATTR_METHODS:
        if method_name not in metrics_summary[model_name]:
            continue
        print(f'  {method_name}:')
        for k_percent in CERT_CONFIG['k_percents']:
            if k_percent not in metrics_summary[model_name][method_name]:
                continue
            m = metrics_summary[model_name][method_name][k_percent]['mean']
            num_images = len(metrics_summary[model_name][method_name][k_percent]['per_image'])
            print(
                f'    K={k_percent}% (n={num_images}): '
                f'Certified={m["pct_certified"]:.1f}%, '
                f'Abstained={m["pct_abstained"]:.1f}%, '
                f'Radius={m["certified_radius"]:.4f}'
            )


METRICS TABLE (per-image aggregated)

densenet121:
  IntegratedGradients:
    K=50% (n=1): Certified=0.9%, Abstained=99.1%, Radius=0.1012
    K=25% (n=1): Certified=11.5%, Abstained=88.5%, Radius=0.1012
    K=5% (n=1): Certified=91.3%, Abstained=8.7%, Radius=0.1012
  GradCAM:
    K=50% (n=1): Certified=100.0%, Abstained=0.0%, Radius=0.1012
    K=25% (n=1): Certified=100.0%, Abstained=0.0%, Radius=0.1012
    K=5% (n=1): Certified=100.0%, Abstained=0.0%, Radius=0.1012
  RISE:


In [13]:
# Visualize certified artifacts (clean heatmap, SS map, certified mask, sample noisy heatmap)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

viz_dir = CERTIFY_DIR
viz_dir.mkdir(parents=True, exist_ok=True)

if not certification_results:
    print('No certification results available for visualization.')
else:
    first_model = next(iter(certification_results))
    first_method = next(iter(certification_results[first_model]))
    k_results = certification_results[first_model][first_method]

    n_rows = len(CERT_CONFIG['k_percents'])
    fig, axes = plt.subplots(n_rows, 4, figsize=(18, 4 * n_rows))
    axes = np.atleast_2d(axes)

    for row_idx, k_percent in enumerate(CERT_CONFIG['k_percents']):
        entries = k_results.get(k_percent, [])
        if not entries:
            for col in range(4):
                axes[row_idx, col].axis('off')
            axes[row_idx, 0].set_title(f'K={k_percent}% (no data)')
            continue

        res = entries[0]['results']  # visualize first image for this K
        certified_map = res.get('certified_map')
        if certified_map is None:
            for col in range(4):
                axes[row_idx, col].axis('off')
            axes[row_idx, 0].set_title(f'K={k_percent}% (missing certified map)')
            continue

        clean_heat = res.get('heatmap_clean')
        ss_map = res.get('ss_map')
        if clean_heat is None:
            clean_heat = np.zeros_like(certified_map, dtype=float)
        if ss_map is None:
            ss_map = np.zeros_like(certified_map, dtype=float)
        noisy_samples = res.get('sample_noisy_heatmaps', [])
        noisy_heat = noisy_samples[0] if noisy_samples else None

        ax_clean = axes[row_idx, 0]
        im0 = ax_clean.imshow(clean_heat, cmap='inferno')
        ax_clean.set_title(f'Clean heatmap (K={k_percent}%)')
        ax_clean.axis('off')
        fig.colorbar(im0, ax=ax_clean, fraction=0.046, pad=0.04)

        ax_ss = axes[row_idx, 1]
        im1 = ax_ss.imshow(ss_map, cmap='magma', vmin=0, vmax=1)
        ax_ss.set_title('Smoothed sparsified (mean mask)')
        ax_ss.axis('off')
        fig.colorbar(im1, ax=ax_ss, fraction=0.046, pad=0.04)

        # Certified map visualization
        viz_map = np.ones((certified_map.shape[0], certified_map.shape[1], 3))
        viz_map[certified_map == 0] = [1.0, 1.0, 1.0]
        viz_map[certified_map == 1] = [1.0, 0.55, 0.0]
        viz_map[certified_map == -1] = [0.8, 0.8, 0.8]
        ax_cert = axes[row_idx, 2]
        ax_cert.imshow(viz_map)
        ax_cert.set_title('Certified map (1=orange, 0=white, abstain=gray)')
        ax_cert.axis('off')
        legend_patches = [
            mpatches.Patch(color=[1.0, 0.55, 0.0], label='certified 1'),
            mpatches.Patch(color=[1.0, 1.0, 1.0], label='certified 0'),
            mpatches.Patch(color=[0.8, 0.8, 0.8], label='abstain'),
        ]
        ax_cert.legend(handles=legend_patches, loc='lower right', fontsize=8)

        ax_noisy = axes[row_idx, 3]
        if noisy_heat is not None:
            im3 = ax_noisy.imshow(noisy_heat, cmap='inferno')
            fig.colorbar(im3, ax=ax_noisy, fraction=0.046, pad=0.04)
            ax_noisy.set_title('Sample noisy heatmap')
        else:
            ax_noisy.text(0.5, 0.5, 'No noisy sample stored', ha='center', va='center')
            ax_noisy.set_title('Sample noisy heatmap')
        ax_noisy.axis('off')

    plt.suptitle(f'{first_model} / {first_method}: certified artifacts', fontsize=14)
    plt.tight_layout()

    viz_path = viz_dir / f'{first_model}_{first_method}_certified_artifacts.png'
    plt.savefig(viz_path, dpi=150, bbox_inches='tight')
    print(f'✓ Saved artifact visualization: {viz_path}')
    plt.close()

✓ Saved artifact visualization: D:\git projects\certified-attribution-medical-imaging\notebooks\output\certifications\isic\densenet121_IntegratedGradients_certified_artifacts.png


# Paper Style Visualization


In [22]:
# Paper-style visualization: Input | SS | K=50% | K=25% | K=5% | Overlayed
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

viz_dir = CERTIFY_DIR
viz_dir.mkdir(parents=True, exist_ok=True)

if not certification_results:
    print('No certification results available for visualization.')
else:
    print(f'Found {len(certification_results)} model(s) in results')
    print(f'Models: {list(certification_results.keys())}')
    
    # Loop over ALL models and create one paper-style visualization per model
    models_processed = 0
    for model_name in certification_results:
        print(f'\nCreating paper-style visualization for {model_name}...')
        
        # Get all methods for this model
        methods_to_show = [m for m in ATTR_METHODS if m in certification_results[model_name]]
        
        if not methods_to_show:
            print(f'  ⚠ No methods found for {model_name}')
            continue
        
        print(f'  Found {len(methods_to_show)} methods: {methods_to_show}')
        
        # Get the test image (we'll use first image from first method's first K)
        first_method = methods_to_show[0]
        first_k = CERT_CONFIG['k_percents'][0]
        
        if first_k not in certification_results[model_name][first_method]:
            print(f'  ⚠ No K={first_k}% results for {model_name}/{first_method}')
            continue
        
        entries = certification_results[model_name][first_method][first_k]
        if not entries:
            print(f'  ⚠ No entries for {model_name}/{first_method}/K={first_k}%')
            continue
        
        print(f'  Found {len(entries)} image entries')
        
        # Get the first image index that was certified
        first_image_idx = entries[0].get('image_idx', 0)
        print(f'  Using image index: {first_image_idx}')
        
        # Get the actual test image from dataset
        test_image_tensor = None
        for img_idx, batch in enumerate(val_loader):
            if img_idx == first_image_idx:
                test_image_tensor = batch['image']
                break
        
        if test_image_tensor is None:
            print(f'  ⚠ Could not load image {first_image_idx} from dataset')
            continue
        
        # Create figure: rows=methods, cols=6 (Input, SS, K=50%, K=25%, K=5%, Overlayed)
        n_rows = len(methods_to_show)
        n_cols = 6
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 3.5 * n_rows))
        if n_rows == 1:
            axes = axes.reshape(1, -1)
        
        for row_idx, method_name in enumerate(methods_to_show):
            # Column 0: Input image
            ax_input = axes[row_idx, 0]
            if test_image_tensor is not None:
                img_np = test_image_tensor.squeeze(0).permute(1, 2, 0).cpu().numpy()
                ax_input.imshow(img_np)
            if row_idx == 0:
                ax_input.set_title('Input', fontsize=12, fontweight='bold')
            # Method name label on the left
            ax_input.text(-0.1, 0.5, method_name, transform=ax_input.transAxes,
                         fontsize=11, fontweight='bold', va='center', ha='right', 
                         rotation=0)
            ax_input.axis('off')
            
            # Get results for this method at different K values
            k_results = {}
            for k in CERT_CONFIG['k_percents']:
                if k in certification_results[model_name][method_name]:
                    entries_k = certification_results[model_name][method_name][k]
                    # Find the entry matching the first_image_idx
                    for entry in entries_k:
                        if entry.get('image_idx') == first_image_idx:
                            k_results[k] = entry['results']
                            break
            
            # Column 1: SS (Smoothed Sparsified) - black and white
            ax_ss = axes[row_idx, 1]
            if 50 in k_results and 'ss_map' in k_results[50]:
                ss_map = k_results[50]['ss_map']
                ax_ss.imshow(ss_map, cmap='gray', vmin=0, vmax=1)
            if row_idx == 0:
                ax_ss.set_title('SS', fontsize=12, fontweight='bold')
            ax_ss.axis('off')
            
            # Columns 2-4: Certified maps for K=50%, 25%, 5%
            for col_idx, k in enumerate([50, 25, 5]):
                ax = axes[row_idx, col_idx + 2]
                if k in k_results:
                    certified_map = k_results[k].get('certified_map')
                    if certified_map is not None:
                        # Color-code: certified-1=orange, certified-0=white, abstain=gray
                        viz_map = np.ones((certified_map.shape[0], certified_map.shape[1], 3))
                        viz_map[certified_map == 0] = [1.0, 1.0, 1.0]  # white
                        viz_map[certified_map == 1] = [1.0, 0.65, 0.0]  # orange
                        viz_map[certified_map == -1] = [0.85, 0.85, 0.85]  # light gray
                        ax.imshow(viz_map)
                if row_idx == 0:
                    ax.set_title(f'K={k}%', fontsize=12, fontweight='bold')
                ax.axis('off')
            
            # Column 5: Overlayed (combine all K certified pixels)
            ax_overlay = axes[row_idx, 5]
            if k_results:
                # Start with gray background
                h, w = next(iter(k_results.values()))['certified_map'].shape
                overlay_map = np.ones((h, w, 3)) * 0.9  # light gray background
                
                # Overlay: K=5% (black/dark), K=25% (red), K=50% (orange)
                # Priority: lower K overwrites higher K
                if 50 in k_results:
                    cert_50 = k_results[50].get('certified_map')
                    if cert_50 is not None:
                        overlay_map[cert_50 == 1] = [1.0, 0.65, 0.0]  # orange for top 50%
                
                if 25 in k_results:
                    cert_25 = k_results[25].get('certified_map')
                    if cert_25 is not None:
                        overlay_map[cert_25 == 1] = [0.9, 0.3, 0.3]  # red for top 25%
                
                if 5 in k_results:
                    cert_5 = k_results[5].get('certified_map')
                    if cert_5 is not None:
                        overlay_map[cert_5 == 1] = [0.2, 0.0, 0.2]  # dark purple for top 5%
                
                ax_overlay.imshow(overlay_map)
            if row_idx == 0:
                ax_overlay.set_title('Overlayed', fontsize=12, fontweight='bold')
            ax_overlay.axis('off')
        
        # Add "Certified" header above K columns
        fig.text(0.63, 0.97, 'Certified', ha='center', fontsize=13, fontweight='bold')
        
        # Add legend
        legend_patches = [
            mpatches.Patch(color=[1.0, 0.65, 0.0], label='Top 50%'),
            mpatches.Patch(color=[0.9, 0.3, 0.3], label='Top 25%'),
            mpatches.Patch(color=[0.2, 0.0, 0.2], label='Top 5%'),
            mpatches.Patch(color=[0.9, 0.9, 0.9], label='Not certified'),
        ]
        fig.legend(handles=legend_patches, loc='lower center', ncol=4, 
                  bbox_to_anchor=(0.5, -0.02), fontsize=10)
        
        plt.suptitle(f'{model_name}: Certified Attributions (Paper Style) - Image {first_image_idx}', 
                    fontsize=14, fontweight='bold', y=0.98)
        plt.tight_layout(rect=[0, 0.02, 1, 0.96])
        
        viz_path = viz_dir / f'{model_name}_img{first_image_idx}_paper_style.png'
        plt.savefig(viz_path, dpi=150, bbox_inches='tight')
        print(f'  ✓ Saved: {viz_path.name}')
        plt.close()
        models_processed += 1
    
    print(f'\n✓ Generated paper-style visualizations for {models_processed}/{len(certification_results)} model(s)')

Found 1 model(s) in results
Models: ['densenet121']

Creating paper-style visualization for densenet121...
  Found 3 methods: ['IntegratedGradients', 'GradCAM', 'RISE']
  Found 1 image entries
  Using image index: 0
  ✓ Saved: densenet121_img0_paper_style.png

✓ Generated paper-style visualizations for 1/1 model(s)


# Figure 4 Style Visualization (5 Images × All Methods)


In [ ]:
# Figure 4 style visualization: rows=images (5), cols=methods with SS and Certified pairs
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

viz_dir = CERTIFY_DIR
viz_dir.mkdir(parents=True, exist_ok=True)

if not certification_results:
    print('No certification results available for visualization.')
else:
    print(f'Creating Figure 4 style visualizations for {len(certification_results)} model(s)...')
    
    # Loop over all models
    for model_name in certification_results:
        print(f'\nProcessing {model_name}...')
        
        # Collect up to 5 images with complete results for all methods
        methods_order = ATTR_METHODS
        image_data_list = []
        
        # First, identify which images have been certified
        # Use first method to find certified images
        first_method = methods_order[0]
        if first_method not in certification_results[model_name]:
            print(f'  ⚠ No results for {first_method}')
            continue
        
        # Get K=50% results to identify certified images
        first_k = CERT_CONFIG['k_percents'][0]
        if first_k not in certification_results[model_name][first_method]:
            print(f'  ⚠ No K={first_k}% results')
            continue
        
        entries = certification_results[model_name][first_method][first_k]
        certified_image_indices = [e.get('image_idx', 0) for e in entries][:5]
        
        if not certified_image_indices:
            print(f'  ⚠ No certified images found')
            continue
        
        print(f'  Found {len(certified_image_indices)} certified images: {certified_image_indices}')
        
        # For each certified image, collect results from all methods
        for img_idx in certified_image_indices:
            # Get the actual image from dataset
            test_image_tensor = None
            for idx, batch in enumerate(val_loader):
                if idx == img_idx:
                    test_image_tensor = batch['image']
                    break
            
            if test_image_tensor is None:
                continue
            
            # Collect results for all methods and K values for this image
            method_results_map = {}
            has_all_methods = True
            
            for method_name in methods_order:
                if method_name not in certification_results[model_name]:
                    has_all_methods = False
                    break
                
                results_by_k = {}
                for k in CERT_CONFIG['k_percents']:
                    if k not in certification_results[model_name][method_name]:
                        continue
                    
                    entries_k = certification_results[model_name][method_name][k]
                    for entry in entries_k:
                        if entry.get('image_idx') == img_idx:
                            results_by_k[k] = entry['results']
                            break
                
                if results_by_k:
                    method_results_map[method_name] = results_by_k
            
            if has_all_methods and len(method_results_map) == len(methods_order):
                image_data_list.append({
                    'image_tensor': test_image_tensor,
                    'method_results_map': method_results_map,
                    'img_idx': img_idx
                })
        
        if not image_data_list:
            print(f'  ⚠ No complete image data found')
            continue
        
        print(f'  Creating visualization with {len(image_data_list)} images...')
        
        # Create the figure
        n_rows = len(image_data_list)
        n_cols = len(methods_order) * 2  # SS and Certified for each method
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.5 * n_cols, 3.5 * n_rows))
        
        if n_rows == 1:
            axes = axes.reshape(1, -1)
        
        for row_idx, img_data in enumerate(image_data_list):
            img_tensor = img_data['image_tensor']
            img_np = img_tensor.squeeze(0).permute(1, 2, 0).cpu().numpy()
            method_results_map = img_data['method_results_map']
            
            for method_idx, method_name in enumerate(methods_order):
                res_by_k = method_results_map.get(method_name, {})
                col_ss = method_idx * 2
                col_cert = method_idx * 2 + 1
                
                # SS column
                ax_ss = axes[row_idx, col_ss]
                ss_map = None
                if res_by_k:
                    ss_map = next(iter(res_by_k.values())).get('ss_map')
                ax_ss.imshow(ss_map if ss_map is not None else np.zeros(img_np.shape[:2]), cmap='gray', vmin=0, vmax=1)
                if row_idx == 0:
                    ax_ss.set_title(f'{method_name}\nSS', fontsize=10, fontweight='bold')
                ax_ss.axis('off')
                
                # Certified column (overlayed K values)
                ax_cert = axes[row_idx, col_cert]
                c50 = res_by_k.get(50, {}).get('certified_map')
                c25 = res_by_k.get(25, {}).get('certified_map')
                c5 = res_by_k.get(5, {}).get('certified_map')
                
                if c50 is None:
                    c50 = np.zeros(img_np.shape[:2])
                if c25 is None:
                    c25 = np.zeros(img_np.shape[:2])
                if c5 is None:
                    c5 = np.zeros(img_np.shape[:2])
                
                # Create overlay map with priority: K=5% > K=25% > K=50%
                overlay_map = np.ones(img_np.shape[:2] + (3,)) * 0.9  # light gray background
                overlay_map[c50 == 1] = [1.0, 0.65, 0.0]  # orange for top 50%
                overlay_map[c25 == 1] = [0.9, 0.3, 0.3]  # red for top 25%
                overlay_map[c5 == 1] = [0.2, 0.0, 0.2]  # dark purple for top 5%
                
                ax_cert.imshow(overlay_map)
                if row_idx == 0:
                    ax_cert.set_title('Certified', fontsize=10, fontweight='bold')
                ax_cert.axis('off')
        
        # Add legend
        legend_patches = [
            mpatches.Patch(color=[1.0, 0.65, 0.0], label='Top 50%'),
            mpatches.Patch(color=[0.9, 0.3, 0.3], label='Top 25%'),
            mpatches.Patch(color=[0.2, 0.0, 0.2], label='Top 5%'),
            mpatches.Patch(color=[0.9, 0.9, 0.9], label='Not certified'),
        ]
        fig.legend(handles=legend_patches, loc='lower center', ncol=4, 
                  bbox_to_anchor=(0.5, -0.01), fontsize=11)
        
        plt.suptitle(f'{model_name}: Overlayed Certified Attributions (Figure 4 Style)', 
                    fontsize=16, fontweight='bold', y=0.995)
        plt.tight_layout(rect=[0, 0.01, 1, 0.99])
        
        viz_path = viz_dir / f'{model_name}_figure4_style.png'
        plt.savefig(viz_path, dpi=150, bbox_inches='tight')
        print(f'  ✓ Saved: {viz_path.name}')
        plt.close()
    
    print(f'\n✓ Generated Figure 4 style visualizations for all models')

In [14]:
# Plot metrics: %Certified vs K and Radius vs K (means)
if 'viz_dir' not in globals():
    viz_dir = CERTIFY_DIR
    viz_dir.mkdir(parents=True, exist_ok=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors = plt.cm.tab10(np.linspace(0, 1, len(ATTR_METHODS)))

if 'metrics_summary' not in globals():
    metrics_summary = compute_metrics_summary(certification_results)

plotted_model = None
for model_idx, model_name in enumerate(list(metrics_summary.keys())[:1]):
    plotted_model = model_name
    for method_idx, method_name in enumerate(ATTR_METHODS):
        if method_name not in metrics_summary[model_name]:
            continue

        k_values = [k for k in CERT_CONFIG['k_percents'] if k in metrics_summary[model_name][method_name]]
        if not k_values:
            continue
        pct_certs = [metrics_summary[model_name][method_name][k]['mean']['pct_certified'] for k in k_values]
        radii = [metrics_summary[model_name][method_name][k]['mean']['certified_radius'] for k in k_values]

        ax1.plot(k_values, pct_certs, marker='o', label=method_name, color=colors[method_idx], linewidth=2)
        ax2.plot(k_values, radii, marker='s', label=method_name, color=colors[method_idx], linewidth=2)

ax1.set_xlabel('Sparsification K (%)', fontsize=12)
ax1.set_ylabel('% Certified (mean)', fontsize=12)
ax1.set_title(f'Certification Rate vs K ({plotted_model or "n/a"})')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.invert_xaxis()  # Higher K on left

ax2.set_xlabel('Sparsification K (%)', fontsize=12)
ax2.set_ylabel('Certified Radius R (mean)', fontsize=12)
ax2.set_title(f'Certified Radius vs K ({plotted_model or "n/a"})')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.invert_xaxis()

plt.tight_layout()

model_for_path = plotted_model or 'no_model'
plot_path = viz_dir / f'{model_for_path}_metrics_plots.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
print(f'✓ Saved metrics plots: {plot_path}')
plt.close()

✓ Saved metrics plots: D:\git projects\certified-attribution-medical-imaging\notebooks\output\certifications\isic\densenet121_metrics_plots.png


In [15]:
# GridPG Evaluation: Localization on 2×2 synthetic grids (Eq. 8)
from src.certify.evaluate import evaluate_certified

gridpg_results = {}

# Create 2×2 synthetic grids from white noise
m = 2
num_grids = 10  # Number of grids to evaluate

if not certification_results:
    print('No certification results available; skipping GridPG evaluation.')
else:
    first_model = next(iter(certification_results))
    first_method = next(iter(certification_results[first_model]))
    k_candidates = sorted(list(certification_results[first_model][first_method].keys()))
    if not k_candidates or not certification_results[first_model][first_method][k_candidates[0]]:
        print('No certified maps found for GridPG visualization.')
    else:
        first_k = k_candidates[0]
        certified_map = certification_results[first_model][first_method][first_k][0]['results']['certified_map']

        for grid_id in range(num_grids):
            # Create m×m grid from random images
            grid_images = [torch.randn(1, 3, 224, 224, device=DEVICE).clamp(0, 1) for _ in range(m*m)]
            grid_img = torch.cat([
                torch.cat([grid_images[i*m + j] for j in range(m)], dim=-1)
                for i in range(m)
            ], dim=-2)  # [1, 3, 224*m, 224*m]
            
            # Tile certified map for grid
            grid_certified = np.tile(certified_map, (m, m))
            
            # Evaluate: target cell at (0, 0)
            cell_h, cell_w = certified_map.shape
            grid_certified_1 = np.sum(grid_certified == 1)
            target_cell = grid_certified[0:cell_h, 0:cell_w]
            target_cell_1 = np.sum(target_cell == 1)
            
            gridpg_score = target_cell_1 / (grid_certified_1 + 1e-8) if grid_certified_1 > 0 else 0.0
            
            gridpg_results[grid_id] = {
                'gridpg_score': float(gridpg_score),
                'target_certified_1': int(target_cell_1),
                'total_certified_1': int(grid_certified_1)
            }

        # Summary
        print('\n' + '='*80)
        print('GRIDPG EVALUATION (Eq. 8)')
        print('='*80)
        print(f'Grid size: {m}×{m}')
        print(f'Number of grids: {num_grids}')
        print(f'Target cell: (0, 0)')
        
        gridpg_scores = [gridpg_results[gid]['gridpg_score'] for gid in range(num_grids)]
        print(f'Mean GridPG score: {np.mean(gridpg_scores):.4f} ± {np.std(gridpg_scores):.4f}')
        print(f'Min: {np.min(gridpg_scores):.4f}, Max: {np.max(gridpg_scores):.4f}')

ImportError: cannot import name 'evaluate_certified' from 'src.certify.evaluate' (D:\git projects\certified-attribution-medical-imaging\src\certify\evaluate.py)

In [ ]:
# Results summary
print('\n' + '='*80)
print('CERTIFICATION PIPELINE - SUMMARY')
print('='*80)

print(f'\n📊 MODELS CERTIFIED:')
for model_name in sorted(certification_results.keys()):
    methods = len(certification_results[model_name])
    print(f'  ✓ {model_name} ({methods} attribution methods)')

print(f'\n🎨 ATTRIBUTION METHODS:')
for method in ATTR_METHODS:
    print(f'  ✓ {method}')

print(f'\n📈 SPARSIFICATION LEVELS:')
for k in CERT_CONFIG['k_percents']:
    print(f'  ✓ K={k}%')

print(f'\n💾 SAVED FILES:')
if CERTIFY_DIR.exists():
    pkl_files = list(CERTIFY_DIR.glob('results_*.pkl'))
    json_files = list(CERTIFY_DIR.glob('results_*.json'))
    print(f'  ✓ Pickle results: {len(pkl_files)} file(s)')
    print(f'  ✓ JSON summary: {len(json_files)} file(s)')

print(f'\n📸 VISUALIZATIONS:')
if CERTIFY_DIR.exists():
    png_files = list(CERTIFY_DIR.glob('*.png'))
    print(f'  ✓ Generated {len(png_files)} visualization(s)')

print(f'\n✅ CERTIFICATION PIPELINE COMPLETE!')
print('='*80)

In [ ]:
# Optional: Detailed metrics table (export to CSV)
import pandas as pd

if 'metrics_summary' not in globals():
    metrics_summary = compute_metrics_summary(certification_results)

if 'viz_dir' not in globals():
    viz_dir = CERTIFY_DIR
    viz_dir.mkdir(parents=True, exist_ok=True)

if 'timestamp' not in globals():
    from datetime import datetime
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# Create detailed metrics dataframe (per-image rows)
metrics_data = []

for model_name in sorted(certification_results.keys()):
    for method_name in ATTR_METHODS:
        if method_name not in certification_results[model_name]:
            continue
        for k_percent in CERT_CONFIG['k_percents']:
            if k_percent not in certification_results[model_name][method_name]:
                continue
            per_image = metrics_summary[model_name][method_name][k_percent]['per_image']
            for entry in per_image:
                metrics_data.append({
                    'Model': model_name,
                    'Method': method_name,
                    'K (%)': k_percent,
                    'Image Idx': entry['image_idx'],
                    'Label': entry['label'],
                    'Certified (%)': f"{entry['pct_certified']:.2f}",
                    'Abstained (%)': f"{entry['pct_abstained']:.2f}",
                    'Certified-1 (%)': f"{entry['pct_certified_1']:.2f}",
                    'Certified-0 (%)': f"{entry['pct_certified_0']:.2f}",
                    'Radius': f"{entry['certified_radius']:.4f}"
                })

df_metrics = pd.DataFrame(metrics_data)

# Save to CSV
csv_path = viz_dir / f'metrics_summary_{timestamp}.csv'
df_metrics.to_csv(csv_path, index=False)
print(f'✓ Saved metrics CSV: {csv_path}')

# Display table (first 10 rows)
print('\nDETAILED METRICS TABLE:')
print(df_metrics.head(10).to_string(index=False))

In [ ]:
# Final status
print('\n' + '='*80)
print('EXECUTION COMPLETE')
print('='*80)
print(f'Started: Cell 1')
print(f'Ended: Cell {14}')
print(f'Timestamp: {timestamp}')
print(f'Total models: {len(certification_results)}')
print(f'Total configs: {len(certification_results) * len(ATTR_METHODS) * len(CERT_CONFIG["k_percents"])} (est.)')
print(f'\nAll outputs saved to: notebooks/output/')
print('='*80)